In [3]:
# Load train/val/test splits produced in 01_eda.ipynb (Steps 1-5: drop, impute, leakage audit, split)
import pandas as pd

processed_dir = r"P:\AI\Project_Default\data\processed"

train_df = pd.read_parquet(f"{processed_dir}\\train.parquet")
val_df = pd.read_parquet(f"{processed_dir}\\val.parquet")
test_df = pd.read_parquet(f"{processed_dir}\\test.parquet")

X_train, y_train = train_df.drop(columns=['target']), train_df['target']
X_val, y_val = val_df.drop(columns=['target']), val_df['target']
X_test, y_test = test_df.drop(columns=['target']), test_df['target']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (964040, 71), Val: (206580, 71), Test: (206581, 71)


In [4]:
# Feature engineering - 5 origination-time features (see notes/02_feat_engineer.md).
# Applied identically to train/val/test via a shared function - no fitting/statistics
# learned from data here, so no train-only-fit concern for these particular features.
import numpy as np

def add_features(df):
    df = df.copy()

    # 1. Loan payment as a share of monthly income
    df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12)

    # 2. Loan size relative to income
    df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

    # 3. Credit history length in months (earliest_cr_line to issue_d)
    earliest_cr_line = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    issue_d = pd.to_datetime(df['issue_d'], format='%b-%Y')
    df['credit_history_length'] = (
        (issue_d.dt.year - earliest_cr_line.dt.year) * 12
        + (issue_d.dt.month - earliest_cr_line.dt.month)
    )

    # 4. Revolving credit headroom relative to limit (inf when total_rev_hi_lim == 0).
    # Use np.nan (not pd.NA) so the column stays float64 rather than becoming object dtype.
    df['avail_credit_ratio'] = df['bc_open_to_buy'] / df['total_rev_hi_lim']
    df['avail_credit_ratio'] = df['avail_credit_ratio'].replace([np.inf, -np.inf], np.nan)

    # 5. Combined FICO score
    df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

    return df

X_train = add_features(X_train)
X_val = add_features(X_val)
X_test = add_features(X_test)

new_feature_cols = [
    'installment_to_income', 'loan_to_income', 'credit_history_length',
    'avail_credit_ratio', 'fico_avg',
]
print(X_train[new_feature_cols].dtypes)
print(X_train[new_feature_cols].describe())


installment_to_income    float64
loan_to_income           float64
credit_history_length      int32
avail_credit_ratio       float64
fico_avg                 float64
dtype: object
       installment_to_income  loan_to_income  credit_history_length  \
count          964040.000000   964040.000000          964040.000000   
mean                0.081203        0.220389             194.972454   
std                 0.434933        1.197056              90.151717   
min                 0.000179        0.000436              12.000000   
25%                 0.046381        0.125000             134.000000   
50%                 0.072299        0.200000             177.000000   
75%                 0.105587        0.291667             240.000000   
max               282.569231      820.512821             999.000000   

       avail_credit_ratio       fico_avg  
count       915885.000000  964040.000000  
mean             0.273580     698.047105  
std              0.230542      31.744971  
min      

In [5]:
# Clip edge cases in the 3 problematic features - bounds computed on X_train only,
# then applied identically to X_train/X_val/X_test (train-only-fit, avoids leakage
# from val/test statistics; see notes/02_feat_engineer.md edge-case note).
clip_cols = ['installment_to_income', 'loan_to_income', 'credit_history_length']
clip_bounds = {col: X_train[col].quantile(0.99) for col in clip_cols}

for df in (X_train, X_val, X_test):
    for col, upper in clip_bounds.items():
        df[col] = df[col].clip(upper=upper)
    # avail_credit_ratio: NaN from the inf-replacement (total_rev_hi_lim == 0) means
    # "no revolving limit to compare against" - fill with 0 (no available headroom)
    df['avail_credit_ratio'] = df['avail_credit_ratio'].fillna(0)

print("Clip bounds (99th percentile on X_train):")
print(clip_bounds)
print()
print(X_train[new_feature_cols].describe())


Clip bounds (99th percentile on X_train):
{'installment_to_income': np.float64(0.201453), 'loan_to_income': np.float64(0.5), 'credit_history_length': np.float64(478.0)}

       installment_to_income  loan_to_income  credit_history_length  \
count          964040.000000   964040.000000          964040.000000   
mean                0.079000        0.214037             194.422636   
std                 0.042632        0.114153              88.103254   
min                 0.000179        0.000436              12.000000   
25%                 0.046381        0.125000             134.000000   
50%                 0.072299        0.200000             177.000000   
75%                 0.105587        0.291667             240.000000   
max                 0.201453        0.500000             478.000000   

       avail_credit_ratio       fico_avg  
count       964040.000000  964040.000000  
mean             0.259915     698.047105  
std              0.232479      31.744971  
min              0

In [11]:
# Save feature-engineered splits to data/processed/ for 03_classifier.ipynb
processed_dir = r"P:\AI\Project_Default\data\processed"

X_train.assign(target=y_train).to_parquet(f"{processed_dir}\\train_fe.parquet", index=False)
X_val.assign(target=y_val).to_parquet(f"{processed_dir}\\val_fe.parquet", index=False)
X_test.assign(target=y_test).to_parquet(f"{processed_dir}\\test_fe.parquet", index=False)

print("Saved feature-engineered train/val/test to data/processed/")


Saved feature-engineered train/val/test to data/processed/


In [6]:
# Round 2 - test deferred feature ideas incrementally (see notes/02_feat_engineer.md).
# Quick-eval helper: trains a Random Forest (using the tuned params found in
# 03_classifier.ipynb - n_estimators=200 for speed, not 400, since this is a fast
# iterate-and-check loop, not the final model) and reports val ROC-AUC/F1.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

def quick_eval(X_tr, y_tr, X_v, y_v, label):
    cat_cols = X_tr.select_dtypes(include='str').columns.tolist()
    preprocessor = ColumnTransformer(transformers=[
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ], remainder='passthrough')

    pipeline = Pipeline(steps=[
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=200, max_depth=None, min_samples_leaf=50, max_features=0.3,
            class_weight='balanced', n_jobs=-1, random_state=42,
        )),
    ])
    pipeline.fit(X_tr, y_tr)

    proba = pipeline.predict_proba(X_v)[:, 1]
    pred = pipeline.predict(X_v)
    auc = roc_auc_score(y_v, proba)
    f1 = f1_score(y_v, pred)
    print(f"{label}: ROC-AUC={auc:.4f}, F1={f1:.4f}")
    return auc, f1

# Baseline (current 5 features, before any Round 2 additions)
baseline_auc, baseline_f1 = quick_eval(X_train, y_train, X_val, y_val, "Baseline (Round 1 features)")


Baseline (Round 1 features): ROC-AUC=0.7224, F1=0.4493


In [ ]:
# Round 2, Feature 1 - sub_grade_numeric (A1=1...G5=35), persisted as its own column
# rather than only existing inside 03_classifier.ipynb's pipeline ordinal encoder.
sub_grade_order = [
    'A1','A2','A3','A4','A5','B1','B2','B3','B4','B5','C1','C2','C3','C4','C5',
    'D1','D2','D3','D4','D5','E1','E2','E3','E4','E5','F1','F2','F3','F4','F5',
    'G1','G2','G3','G4','G5',
]
sub_grade_map = {sg: i + 1 for i, sg in enumerate(sub_grade_order)}

for df in (X_train, X_val, X_test):
    df['sub_grade_numeric'] = df['sub_grade'].map(sub_grade_map)

print(X_train['sub_grade_numeric'].describe())
print()

step1_auc, step1_f1 = quick_eval(X_train, y_train, X_val, y_val, "Step 1: + sub_grade_numeric")
print(f"Delta vs baseline: ROC-AUC {step1_auc - baseline_auc:+.4f}, F1 {step1_f1 - baseline_f1:+.4f}")


In [8]:
# Round 2, Feature 2 - revol_util_x_dti interaction (high utilization AND high DTI
# compounding risk). Note: a tree can already approximate this via successive splits
# on revol_util then dti (or vice versa), so any gain here would mean the explicit
# product makes that interaction easier/faster for the tree to find, not that it's
# undiscoverable otherwise.
for df in (X_train, X_val, X_test):
    df['revol_util_x_dti'] = df['revol_util'] * df['dti']

print(X_train['revol_util_x_dti'].describe())
print()

step2_auc, step2_f1 = quick_eval(X_train, y_train, X_val, y_val, "Step 2: + revol_util_x_dti")
print(f"Delta vs step 1: ROC-AUC {step2_auc - step1_auc:+.4f}, F1 {step2_f1 - step1_f1:+.4f}")
print(f"Delta vs baseline: ROC-AUC {step2_auc - baseline_auc:+.4f}, F1 {step2_f1 - baseline_f1:+.4f}")


count    964040.000000
mean        988.111909
std         842.762060
min         -78.200000
25%         441.037500
50%         854.652500
75%        1392.940750
max       98101.800000
Name: revol_util_x_dti, dtype: float64

Step 2: + revol_util_x_dti: ROC-AUC=0.7226, F1=0.4493
Delta vs step 1: ROC-AUC -0.0001, F1 -0.0009
Delta vs baseline: ROC-AUC +0.0002, F1 +0.0001


In [9]:
# Sanity check: revol_util_x_dti has a negative min (-78.2), but both revol_util (%)
# and dti (ratio) should be >= 0. Check which one has negative values.
print("revol_util negative count:", (X_train['revol_util'] < 0).sum())
print("revol_util min:", X_train['revol_util'].min())
print()
print("dti negative count:", (X_train['dti'] < 0).sum())
print("dti min:", X_train['dti'].min())
print()
print(X_train.loc[X_train['dti'] < 0, ['revol_util', 'dti', 'revol_util_x_dti']].head())


revol_util negative count: 0
revol_util min: 0.0

dti negative count: 1
dti min: -1.0

        revol_util  dti  revol_util_x_dti
389188        78.2 -1.0             -78.2


In [10]:
# Round 2, Feature 3 - high_utilization_flag (bc_util > 80) - catches a "danger zone"
# nonlinearity that a raw ratio might average out.
for df in (X_train, X_val, X_test):
    df['high_utilization_flag'] = (df['bc_util'] > 80).astype(int)

print(X_train['high_utilization_flag'].value_counts(normalize=True))
print()

step3_auc, step3_f1 = quick_eval(X_train, y_train, X_val, y_val, "Step 3: + high_utilization_flag")
print(f"Delta vs step 2: ROC-AUC {step3_auc - step2_auc:+.4f}, F1 {step3_f1 - step2_f1:+.4f}")
print(f"Delta vs baseline: ROC-AUC {step3_auc - baseline_auc:+.4f}, F1 {step3_f1 - baseline_f1:+.4f}")


high_utilization_flag
0    0.706397
1    0.293603
Name: proportion, dtype: float64

Step 3: + high_utilization_flag: ROC-AUC=0.7227, F1=0.4495
Delta vs step 2: ROC-AUC +0.0001, F1 +0.0001
Delta vs baseline: ROC-AUC +0.0003, F1 +0.0002


In [11]:
# Round 2, Feature 4 - active_account_ratio = num_actv_rev_tl / total_acc - proportion
# of credit history that's currently active/live vs. total accounts ever opened.
for df in (X_train, X_val, X_test):
    df['active_account_ratio'] = df['num_actv_rev_tl'] / df['total_acc']
    df['active_account_ratio'] = df['active_account_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0)

print(X_train['active_account_ratio'].describe())
print()

step4_auc, step4_f1 = quick_eval(X_train, y_train, X_val, y_val, "Step 4: + active_account_ratio")
print(f"Delta vs step 3: ROC-AUC {step4_auc - step3_auc:+.4f}, F1 {step4_f1 - step3_f1:+.4f}")
print(f"Delta vs baseline: ROC-AUC {step4_auc - baseline_auc:+.4f}, F1 {step4_f1 - baseline_f1:+.4f}")


count    964040.000000
mean          0.244005
std           0.156699
min           0.000000
25%           0.134615
50%           0.217391
75%           0.333333
max           1.000000
Name: active_account_ratio, dtype: float64

Step 4: + active_account_ratio: ROC-AUC=0.7225, F1=0.4496
Delta vs step 3: ROC-AUC -0.0002, F1 +0.0001
Delta vs baseline: ROC-AUC +0.0001, F1 +0.0003


In [12]:
# Round 2, Feature 5 - purpose_grouped: collapse 14 purpose categories into fewer,
# better-populated buckets (2 categories cover 77% of rows; several others have only
# a few hundred to a few thousand rows - e.g. educational 252, renewable_energy 636).
purpose_groups = {
    'debt_consolidation': 'debt_related',
    'credit_card': 'debt_related',
    'home_improvement': 'major_purchase',
    'major_purchase': 'major_purchase',
    'house': 'major_purchase',
    'car': 'major_purchase',
    'small_business': 'business',
    'medical': 'discretionary',
    'moving': 'discretionary',
    'vacation': 'discretionary',
    'wedding': 'discretionary',
    'renewable_energy': 'discretionary',
    'educational': 'discretionary',
    'other': 'other',
}

for df in (X_train, X_val, X_test):
    df['purpose_grouped'] = df['purpose'].map(purpose_groups)

print(X_train['purpose_grouped'].value_counts())
print()

step5_auc, step5_f1 = quick_eval(X_train, y_train, X_val, y_val, "Step 5: + purpose_grouped")
print(f"Delta vs step 4: ROC-AUC {step5_auc - step4_auc:+.4f}, F1 {step5_f1 - step4_f1:+.4f}")
print(f"Delta vs baseline: ROC-AUC {step5_auc - baseline_auc:+.4f}, F1 {step5_f1 - baseline_f1:+.4f}")


purpose_grouped
debt_related      770769
major_purchase     99261
other              56113
discretionary      26806
business           11091
Name: count, dtype: int64

Step 5: + purpose_grouped: ROC-AUC=0.7228, F1=0.4503
Delta vs step 4: ROC-AUC +0.0003, F1 +0.0008
Delta vs baseline: ROC-AUC +0.0004, F1 +0.0011
